In [1]:
import os
import modelseedpy

modelseedpy 0.4.2


In [2]:
genome = modelseedpy.MSGenome.from_fasta('../Model Reconstruction/GCF_000005845.2_ASM584v2_protein.faa')

In [3]:
import json
if os.path.exists('../ecoli_rast.json'):
    with open('../ecoli_rast.json', 'r') as fh:
        genome_annotation = json.load(fh)
        for f in genome.features:
            f.ontology_terms = {}
            for s in genome_annotation.get(f.id, []):
                f.add_ontology_term('RAST', s)
else:
    from modelseedpy import RastClient
    rast = RastClient()
    res = rast.annotate_genome(genome)
    genome_annotation = {f.id:list(f.ontology_terms.get('RAST', [])) for f in genome.features}
    with open('../ecoli_rast.json', 'w') as fh:
        fh.write(json.dumps(genome_annotation))

In [4]:
from modelseedpy.core.mspredict import MSPredict
mspredict = MSPredict()
mspredict.genome_classifier.model

KNeighborsClassifier(weights='distance')

In [5]:
genome_class = mspredict.predict(genome)
print(genome_class)
template_core, template_genome_scale = mspredict.auto_select_template(genome_class)

MSGenomeClass.N


In [16]:
from modelseedpy import MSBuilder
builder = MSBuilder(genome, template=template_genome_scale, name='ecoli')

In [19]:
model_base = builder.build('ecoli', annotate_with_rast=False)

1 True True
0 True False
1 True True
1 True True
0 True False
0 True False
0 True False
1 True True
0 True False
0 True False
1 True True
1 True True
0 True False
0 True False
0 True False
1 True True
1 True True
0 True False
0 True False
0 True False
1 True True
1 True True
1 True True
1 True True
0 True False
0 True False
1 True True
0 True False
1 True True
0 True False
1 True True
1 True True
0 True False
1 True True
1 True True
0 True False
1 True True
1 True True
1 True True
1 True True
0 True False
0 True False
0 True False
0 True False
0 True False
0 True False
1 True True
0 True False
1 True True
0 True False
0 True False
1 True True
0 True False
1 True True
0 True False
0 True False
0 True False
1 True True
1 True True
0 True False
0 True False
0 True False
1 True True
1 True True
0 True False
1 True True
0 True False
0 True False
1 True True
0 True False
0 True False
1 True True
0 True False
0 True False
0 True False
0 True False
0 True False
0 True False
1 True True
0 True 

In [17]:
from modelseedpy import MSBuilder, MSATPCorrection, MSMedia, MSGapfill

'D:\\opt\\python\\env\\modelseed\\Lib\\site-packages\\modelseedpy\\core\\msatpcorrection.py'

In [37]:
import os.path as _path
import pandas as pd
from modelseedpy.core.msatpcorrection import min_gap
current_file_path = _path.dirname(_path.abspath(modelseedpy.core.msatpcorrection.__file__))
default_media_path = f"{current_file_path}/../data/atp_medias.tsv"
if os.path.exists(default_media_path):
    medias = pd.read_csv(default_media_path, sep="\t", index_col=0).to_dict()
def load_default_medias(medias):
    res = []
    for media_id in medias:
        #print(media_id)
        media_d = {}
        for exchange, v in medias[media_id].items():
            if v > 0:
                k = exchange.split("_")[1]
                media_d[k] = v
        media_d["cpd00001"] = 1000
        media_d["cpd00067"] = 1000
        media = MSMedia.from_dict(media_d)
        media.id = media_id
        media.name = media_id
        min_obj = 0.01
        res.append([media, min_gap.get(media.id, min_obj)])
    return res
medias_atp = load_default_medias(medias)

In [41]:
atp_correction = MSATPCorrection(model_base, template_core, atp_medias=medias_atp, load_default_medias=False)

Ignoring reaction 'rxn00062_c0' since it already exists.


In [42]:
test = atp_correction.run_atp_correction()

No gapfilling solution found before filtering for Ac.O2 activating rxn00062_c0
No gapfilling solution found before filtering for Etho.O2 activating rxn00062_c0
No gapfilling solution found before filtering for Pyr.O2 activating rxn00062_c0
No gapfilling solution found before filtering for Akg.O2 activating rxn00062_c0
No gapfilling solution found before filtering for Dlac.O2 activating rxn00062_c0
No gapfilling solution found before filtering for For.O2 activating rxn00062_c0
No gapfilling solution found before filtering for Ac activating rxn00062_c0
No gapfilling solution found before filtering for Etho activating rxn00062_c0
No gapfilling solution found before filtering for Pyr activating rxn00062_c0
No gapfilling solution found before filtering for Glyc activating rxn00062_c0
No gapfilling solution found before filtering for Succ activating rxn00062_c0
No gapfilling solution found before filtering for Akg activating rxn00062_c0
No gapfilling solution found before filtering for Dlac 

In [43]:
test

[{'media': <modelseedpy.core.msmedia.MSMedia at 0x2a8d5f74090>,
  'is_max_threshold': True,
  'threshold': 1e-05,
  'objective': 'rxn00062_c0'},
 {'media': <modelseedpy.core.msmedia.MSMedia at 0x2a8d59b49d0>,
  'is_max_threshold': True,
  'threshold': 2.0000000000000018,
  'objective': 'rxn00062_c0'},
 {'media': <modelseedpy.core.msmedia.MSMedia at 0x2a8d61a6510>,
  'is_max_threshold': True,
  'threshold': 1.200000000000001,
  'objective': 'rxn00062_c0'},
 {'media': <modelseedpy.core.msmedia.MSMedia at 0x2a8d5de2750>,
  'is_max_threshold': True,
  'threshold': 1.200000000000001,
  'objective': 'rxn00062_c0'}]

In [8]:
for rxn_id in rxn_gpr:
    for cpx_id in rxn_gpr[rxn_id]:
        template_cpx = template_genome_scale.complexes.get_by_id(cpx_id)
        n_trig = 0
        for role, (trig, optional) in template_cpx.roles.items():
            if role.id in rxn_gpr[rxn_id][cpx_id]:
                if trig:
                    n_trig += 1
        if n_trig < 1:
            print(rxn_id, '!!', cpx_id)

In [31]:
rxn_gpr[rxn_id][cpx_id]

{'ftr01607': {'NP_414684.1'}}

In [9]:
from modelseedpy.core.mstemplate import MSTemplate, NewModelTemplateComplex, NewModelTemplateRole
from modelseedpy.core.mstemplate import MSTemplateSpecies, MSTemplateReaction, MSTemplateMetabolite
template_test = MSTemplate('test')
template_cpx = NewModelTemplateComplex("cpx_test1", "Test Complex")
template_cpx.add_role(NewModelTemplateRole('role_t', 'role_trig'), triggering=True)
template_cpx.add_role(NewModelTemplateRole('role_o', 'role_opt'), triggering=False)
template_test.add_complexes([template_cpx])

template_cpd = MSTemplateMetabolite('cpd1')
template_spi_e = MSTemplateSpecies('cpd1_e', 0, 'e', 'cpd1')
template_spi_c = MSTemplateSpecies('cpd1_c', 0, 'c', 'cpd1')
template_test.add_compounds([template_cpd])
template_test.add_comp_compounds([template_spi_e, template_spi_c])

template_rxn = MSTemplateReaction('rxn1', 'rxn1')
template_rxn.add_metabolites({
    template_test.compcompounds.cpd1_e: -1,
    template_test.compcompounds.cpd1_c: 1
})
template_rxn.add_complexes([template_test.complexes.cpx_test1])
template_test.add_reactions([template_rxn])

In [10]:
from modelseedpy.core.msgenome import MSGenome, MSFeature
feature_t = MSFeature("feature_t", "MKV")
feature_t.add_ontology_term('RAST', 'role_trig')
feature_o = MSFeature("feature_o", "MKV")
feature_o.add_ontology_term('RAST', 'role_opt')
genome1 = MSGenome()
genome1.add_features([feature_t, feature_o])
genome2 = MSGenome()
genome2.add_features([feature_t])
genome3 = MSGenome()
genome3.add_features([feature_o])

In [11]:
from modelseedpy import MSBuilder
builder1 = MSBuilder(genome1, template=template_test, name='test1')
builder2 = MSBuilder(genome2, template=template_test, name='test2')
builder3 = MSBuilder(genome3, template=template_test, name='test3')

In [12]:
grp1 = builder1.generate_reaction_complex_sets()
grp2 = builder2.generate_reaction_complex_sets()
grp3 = builder3.generate_reaction_complex_sets()

1 True True
1 True True
0 True False


In [13]:
grp1

{'rxn1': {'cpx_test1': {'role_t': {'feature_t'}, 'role_o': {'feature_o'}}}}

In [14]:
grp2

{'rxn1': {'cpx_test1': {'role_t': {'feature_t'}}}}

In [15]:
grp3

{}

In [84]:
match_complex = builder3._get_template_reaction_complexes(template_test.reactions.rxn1)

In [83]:
builder3._build_reaction_complex_gpr_sets2(builder3._get_template_reaction_complexes(template_test.reactions.rxn1))

{'cpx_test1': {'role_o': {'feature_o'}}}

In [88]:
match_complex

{'cpx_test1': {'role_t': ['roletrig', True, False, set()],
  'role_o': ['roleopt', False, False, {'feature_o'}]}}

In [120]:

def _build_reaction_complex_gpr_sets2(
    match_complex, allow_incomplete_complexes=True
):
    complexes = {}
    for cpx_id, cpx_roles in match_complex.items():
        #print(cpx_id, cpx_roles)
        complete = True
        trig_roles = {}
        roles = set()
        role_genes = {}
        for role_id, [role_name, trig, opt, feature_ids] in cpx_roles.items():
            t = match_complex[cpx_id][role_id]
            if trig and len(feature_ids) == 0:
                complete = False
            if trig and len(feature_ids) > 0:
                trig_roles[role_id] = t[3]
            #complete &= len(t[3]) > 0 or not t[1] or t[2]
            if len(feature_ids) > 0:
                roles.add(role_id)
                role_genes[role_id] = t[3]
        # print(cpx_id, complete, roles)
        print(len(trig_roles), allow_incomplete_complexes, complete)
        if len(trig_roles) > 0 and (allow_incomplete_complexes or complete):
            complexes[cpx_id] = {}
            for role_id in role_genes:
                complexes[cpx_id][role_id] = role_genes[role_id]
                # print(role_id, role_genes[role_id])
        #print(complete, len(trig_roles) > 0)
    return complexes


In [121]:
print(_build_reaction_complex_gpr_sets2(builder1._get_template_reaction_complexes(template_test.reactions.rxn1)))
print(_build_reaction_complex_gpr_sets2(builder2._get_template_reaction_complexes(template_test.reactions.rxn1)))
print(_build_reaction_complex_gpr_sets2(builder3._get_template_reaction_complexes(template_test.reactions.rxn1)))

1 True True
{'cpx_test1': {'role_t': {'feature_t'}, 'role_o': {'feature_o'}}}
1 True True
{'cpx_test1': {'role_t': {'feature_t'}}}
0 True False
{}


In [122]:
print(_build_reaction_complex_gpr_sets2(builder1._get_template_reaction_complexes(template_test.reactions.rxn1), False))
print(_build_reaction_complex_gpr_sets2(builder2._get_template_reaction_complexes(template_test.reactions.rxn1), False))
print(_build_reaction_complex_gpr_sets2(builder3._get_template_reaction_complexes(template_test.reactions.rxn1), False))

1 False True
{'cpx_test1': {'role_t': {'feature_t'}, 'role_o': {'feature_o'}}}
1 False True
{'cpx_test1': {'role_t': {'feature_t'}}}
0 False False
{}
